In [1]:
import numpy as np
import torch
import torch.nn as nn
import onnx
import onnxruntime as ort
import time
import os
from pathlib import Path

DATA = Path("../data/processed")
MODELS = Path("../models")

class StudentMLP(nn.Module):
    def __init__(self, input_dim=24, hidden=(64, 32, 16)):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU()]
            prev = h
        layers += [nn.Linear(prev, 1)]
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(-1)

student = StudentMLP()
student.load_state_dict(torch.load(MODELS / "student.pt", map_location="cpu"))
student.eval()
print("Student loaded.")

Student loaded.


In [4]:
onnx_path = MODELS / "student.onnx"
dummy = torch.randn(1, 24)

torch.onnx.export(
    student,
    dummy,
    onnx_path,
    input_names=["input"],
    output_names=["prediction"],
    dynamic_axes={"input": {0: "batch"}, "prediction": {0: "batch"}},
    opset_version=17
)

print(f"Exported: {onnx_path}")
print(f"File size: {os.path.getsize(onnx_path) / 1024:.2f} KB")

# Sanity check
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
print("ONNX model is valid.")

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_11504\1638489294.py:4: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W1009 22:04:03.909000 11504 Lib\site-packages\torch\onnx\_internal\exporter\_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W1009 22:04:09.636000 11504 Lib\site-packages\torch\onnx\_internal\exporter\_registration.py:107] torchvision is not installed. Skipping torchvision::nms
W1009 22:04:09.643000

[torch.onnx] Obtain model graph for `StudentMLP([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `StudentMLP([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Exported: ..\models\student.onnx
File size: 2.20 KB
ONNX model is valid.


In [5]:
X_test = np.load(DATA / "elia_X_test.npy").astype(np.float32)

# PyTorch prediction
with torch.no_grad():
    torch_pred = student(torch.tensor(X_test)).numpy()

# ONNX prediction
sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
onnx_pred = sess.run(None, {"input": X_test})[0]

# Compare
max_diff = np.abs(torch_pred - onnx_pred).max()
print(f"Max |PyTorch − ONNX|: {max_diff:.8f}")
print("✅ Match!" if max_diff < 1e-5 else "❌ Mismatch — check export")

Max |PyTorch − ONNX|: 0.00000024
✅ Match!


In [6]:
# Single-sample latency (this is the edge case — one reading at a time)
X_single = X_test[:1]

# PyTorch
with torch.no_grad():
    _ = student(torch.tensor(X_single))   # warmup
    t0 = time.perf_counter()
    for _ in range(1000):
        _ = student(torch.tensor(X_single))
    torch_lat = (time.perf_counter() - t0) / 1000 * 1000

# ONNX
_ = sess.run(None, {"input": X_single})
t0 = time.perf_counter()
for _ in range(1000):
    _ = sess.run(None, {"input": X_single})
onnx_lat = (time.perf_counter() - t0) / 1000 * 1000

print(f"PyTorch latency: {torch_lat:.4f} ms")
print(f"ONNX latency:    {onnx_lat:.4f} ms")
print(f"ONNX speedup:    {torch_lat / onnx_lat:.1f}×")

PyTorch latency: 0.8205 ms
ONNX latency:    0.1978 ms
ONNX speedup:    4.1×


In [7]:
import joblib

X_scaler = joblib.load(MODELS / "scaler.pkl")     # for X features
y_scaler = joblib.load(MODELS / "elia_y_scaler.pkl")

def predict_pipeline(raw_window_mw):
    """
    raw_window_mw: list or array of 24 hourly MW readings (raw, unscaled)
    returns: predicted next-hour MW
    """
    # Scale input
    x_scaled = X_scaler.transform(np.array(raw_window_mw).reshape(1, -1)).astype(np.float32)
    # ONNX inference
    pred_scaled = sess.run(None, {"input": x_scaled})[0]
    # Inverse-scale output
    pred_mw = y_scaler.inverse_transform(pred_scaled.reshape(-1, 1)).flatten()[0]
    return pred_mw

# Test on one real window
sample_window = X_test[0]  # already scaled
# For a true end-to-end test we'd need the raw MW — but scaled pipeline works for latency demo
t0 = time.perf_counter()
for _ in range(100):
    _ = predict_pipeline(sample_window.reshape(1, -1))
end_to_end = (time.perf_counter() - t0) / 100 * 1000

print(f"End-to-end pipeline latency: {end_to_end:.3f} ms per prediction")

End-to-end pipeline latency: 3.365 ms per prediction


In [8]:
import json

onnx_size_kb = os.path.getsize(onnx_path) / 1024
metadata = {
    "onnx_size_kb": onnx_size_kb,
    "pytorch_latency_ms": torch_lat,
    "onnx_latency_ms": onnx_lat,
    "onnx_speedup": torch_lat / onnx_lat,
    "end_to_end_latency_ms": end_to_end,
    "opset_version": 17,
    "input_shape": [1, 24],
    "output_shape": [1],
}
with open(DATA / "onnx_benchmark.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(json.dumps(metadata, indent=2))

{
  "onnx_size_kb": 2.197265625,
  "pytorch_latency_ms": 0.8205229000013787,
  "onnx_latency_ms": 0.19779109999944922,
  "onnx_speedup": 4.148431855648023,
  "end_to_end_latency_ms": 3.364941000036197,
  "opset_version": 17,
  "input_shape": [
    1,
    24
  ],
  "output_shape": [
    1
  ]
}
